# BPA 10-Bus System - Hybrid Python/MATPOWER Notebook
## Sensitivity Matrix - Clustering - VQ Curves

**Sections**
1. Dependencies
2. BPA System Data
3. Base Power Flow (PYPOWER)
4. dFq/dQ Sensitivity Matrix
5. Article Matrix (Table 2) Comparison
6. Heatmaps
7. Clustering Functions
8. Validation Sweep 0% to 10%
9. Support Curve Plot
10. Minimum Observer Analysis
11. MATPOWER VQ Analysis (optional - requires MATLAB)
12. Excel Export

## 1 - Dependencies

In [ ]:
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import numpy as _np_check
    _major = int(_np_check.__version__.split('.')[0])
    if _major >= 2:
        _pip('numpy<2.0')
except Exception:
    pass

for _p in ['pandas', 'matplotlib', 'seaborn', 'openpyxl', 'scipy', 'pypower']:
    try:
        __import__(_p)
    except ImportError:
        _pip(_p)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from IPython.display import display

# Output directory (current directory)
OUT = Path('.')

# PYPOWER column indices
BUS_VM, BUS_VA, BUS_PD, BUS_QD = 7, 8, 2, 3
GEN_BUS, GEN_PG, GEN_QG = 0, 1, 2
F_BUS, T_BUS, PF, QF, PT, QT = 0, 1, 13, 14, 15, 16

from pypower.api import runpf, ppoption

print('All imports OK')


## 2 - BPA 10-Bus System Data

In [ ]:
baseMVA = 100.0

bus = np.array([
    [ 1, 3,    0,    0, 0,   0, 1, 0.9800,   0.0, 500, 1, 1.1, 0.9],
    [ 2, 2,    0,    0, 0,   0, 1, 0.9646,  -8.6, 500, 1, 1.1, 0.9],
    [ 3, 2,    0,    0, 0,   0, 1, 0.9553, -26.8, 500, 1, 1.1, 0.9],
    [ 4, 1,    0,    0, 0,   0, 1, 1.0874,  -4.0, 500, 1, 1.1, 0.9],
    [ 5, 1,    0,    0, 0,   0, 1, 1.0616, -12.4, 500, 1, 1.1, 0.9],
    [ 6, 1,    0,    0, 0, 520, 1, 1.0196, -30.7, 500, 1, 1.1, 0.9],
    [ 7, 1, 2500,  700, 0, 445, 1, 0.9362, -37.2, 500, 1, 1.1, 0.9],
    [ 8, 1,    0,    0, 0, 445, 1, 0.9337, -36.8, 500, 1, 1.1, 0.9],
    [ 9, 1,    0,    0, 0,   0, 1, 0.8850, -44.1, 500, 1, 1.1, 0.9],
    [10, 1, 2500,    0, 0,   0, 1, 0.9220, -46.4, 500, 1, 1.1, 0.9],
], dtype=float)

gen = np.array([
    [1, 4219, 1207, 1600, -1000, 0.9800, 100, 1, 4737.1, 10, 0,0,0,0,0,0,0,0,0,0,0],
    [2, 1736,  725,  725,  -200, 0.9646, 100, 1, 2077.1, 10, 0,0,0,0,0,0,0,0,0,0,0],
    [3, 1155,  700,  700,  -100, 0.9730, 100, 1, 1438.7, 10, 0,0,0,0,0,0,0,0,0,0,0],
], dtype=float)

branch = np.array([
    [ 1,  4, 0.0000, 0.0020, 1.0000, 5000, 5000, 5000, 0.8857, 0, 1, -360, 360],
    [ 2,  5, 0.0000, 0.0045, 1.0000, 2200, 2200, 2200, 0.8857, 0, 1, -360, 360],
    [ 3,  6, 0.0000, 0.0125, 0.0000, 1600, 1600, 1600, 0.9024, 0, 1, -360, 360],
    [ 3,  6, 0.0000, 0.0125, 0.0000, 1600, 1600, 1600, 0.9024, 0, 1, -360, 360],
    [ 4,  5, 0.0000, 0.0040, 0.0000, 5000, 5000, 5000, 0.0000, 0, 1, -360, 360],
    [ 5,  6, 0.0015, 0.0288, 1.1730, 1100, 1100, 1100, 0.0000, 0, 1, -360, 360],
    [ 5,  6, 0.0015, 0.0288, 1.1730, 1100, 1100, 1100, 0.0000, 0, 1, -360, 360],
    [ 5,  6, 0.0015, 0.0288, 1.1730, 1100, 1100, 1100, 0.0000, 0, 1, -360, 360],
    [ 5,  6, 0.0015, 0.0288, 1.1730, 1100, 1100, 1100, 0.0000, 0, 1, -360, 360],
    [ 5,  6, 0.0015, 0.0288, 1.1730, 1100, 1100, 1100, 0.0000, 0, 1, -360, 360],
    [ 6,  7, 0.0000, 0.0030, 0.0000, 3600, 3600, 3600, 1.0664, 0, 1, -360, 360],
    [ 6,  8, 0.0000, 0.0026, 0.0000, 3500, 3500, 3500, 1.0800, 0, 1, -360, 360],
    [ 8,  9, 0.0010, 0.0030, 0.0000, 3800, 3800, 3800, 0.0000, 0, 1, -360, 360],
    [ 9, 10, 0.0000, 0.0010, 0.0000, 3700, 3700, 3700, 0.9600, 0, 1, -360, 360],
], dtype=float)

gencost = np.array([
    [2, 1500, 0, 3, 0.1100, 5.0, 150],
    [2, 2000, 0, 3, 0.0850, 1.2, 600],
    [2, 3000, 0, 3, 0.1225, 1.0, 335],
], dtype=float)

EVALUATED_BUSES = [4, 5, 6, 7, 8, 9, 10]

def make_branch_names(br):
    counts = {}
    names = []
    for row in br:
        key = (int(row[0]), int(row[1]))
        counts[key] = counts.get(key, 0) + 1
        suffix = '' if counts[key] == 1 else f'_{counts[key]}'
        names.append(f'B{key[0]}-B{key[1]}{suffix}')
    return names

BRANCH_NAMES = make_branch_names(branch)
print('Branch names:', BRANCH_NAMES)

def build_ppc(bus_arr=None, gen_arr=None, branch_arr=None, gencost_arr=None):
    return {
        'version': '2',
        'baseMVA': baseMVA,
        'bus':     (bus_arr     if bus_arr     is not None else bus    ).copy(),
        'gen':     (gen_arr     if gen_arr     is not None else gen    ).copy(),
        'branch':  (branch_arr  if branch_arr  is not None else branch ).copy(),
        'gencost': (gencost_arr if gencost_arr is not None else gencost).copy(),
    }

print('build_ppc ready')


## 3 - System DataFrames

In [ ]:
BUS_COLS    = ['Bus','Type','Pd','Qd','Gs','Bs','Area','Vm','Va','BaseKV','Zone','Vmax','Vmin']
GEN_COLS    = ['Bus','Pg','Qg','Qmax','Qmin','Vg','mBase','Status','Pmax','Pmin',
               'Pc1','Pc2','Qc1min','Qc1max','Qc2min','Qc2max','ramp_agc',
               'ramp_10','ramp_30','ramp_q','apf']
BRANCH_COLS = ['fbus','tbus','r','x','b','rateA','rateB','rateC','ratio','angle','status','angmin','angmax']

df_bus    = pd.DataFrame(bus,    columns=BUS_COLS).set_index('Bus')
df_gen    = pd.DataFrame(gen,    columns=GEN_COLS[:gen.shape[1]]).set_index('Bus')
df_branch = pd.DataFrame(branch, columns=BRANCH_COLS, index=BRANCH_NAMES)

print('=== Bus Data ===')
display(df_bus)
print('=== Generator Data ===')
display(df_gen)
print('=== Branch Data ===')
display(df_branch)


## 4 - Base Power Flow (PYPOWER)

In [ ]:
opt = ppoption(VERBOSE=0, OUT_ALL=0, ENFORCE_Q_LIMS=0)
ppc_base = build_ppc()
res_base = runpf(ppc_base, opt)

if not res_base[1]:
    raise RuntimeError('Base power flow did not converge!')

bus_res    = res_base[0]['bus']
gen_res    = res_base[0]['gen']
branch_res = res_base[0]['branch']

df_vm = pd.DataFrame({
    'Bus':     bus_res[:, 0].astype(int),
    'Vm(pu)':  np.round(bus_res[:, BUS_VM], 4),
    'Va(deg)': np.round(bus_res[:, BUS_VA], 4),
}).set_index('Bus')
print('Bus Voltages:')
display(df_vm)

df_qg = pd.DataFrame({
    'Gen Bus':  gen_res[:, GEN_BUS].astype(int),
    'Qg(MVAR)': np.round(gen_res[:, GEN_QG], 3),
}).set_index('Gen Bus')
print('Generator Reactive Output:')
display(df_qg)

df_qf = pd.DataFrame({
    'Branch':   BRANCH_NAMES,
    'QF(MVAR)': np.round(branch_res[:, QF], 3),
    'QT(MVAR)': np.round(branch_res[:, QT], 3),
}).set_index('Branch')
print('Branch Reactive Flows:')
display(df_qf)


## 5 - dFq/dQ Sensitivity Matrix (Finite Differences)

In [ ]:
def compute_sensitivity(ppc_in, eval_buses, delta_mvar=1.0):
    opt_s = ppoption(VERBOSE=0, OUT_ALL=0, ENFORCE_Q_LIMS=0)
    res0  = runpf(ppc_in, opt_s)
    if not res0[1]:
        raise RuntimeError('Base PF for sensitivity did not converge')
    Qf_base = res0[0]['branch'][:, QF].copy()

    n_br   = ppc_in['branch'].shape[0]
    S_data = np.zeros((n_br, len(eval_buses)))

    for j_idx, bus_num in enumerate(eval_buses):
        ppc_p = build_ppc(
            bus_arr    = ppc_in['bus'].copy(),
            gen_arr    = ppc_in['gen'].copy(),
            branch_arr = ppc_in['branch'].copy(),
        )
        # bus_num is 1-based; array row index = bus_num - 1
        b_idx = int(bus_num) - 1
        ppc_p['bus'][b_idx, BUS_QD] -= delta_mvar  # decrease Qd = increase injection
        res_p = runpf(ppc_p, opt_s)
        if not res_p[1]:
            print(f'  WARNING: PF did not converge for bus {bus_num} perturbation')
            continue
        Qf_p = res_p[0]['branch'][:, QF]
        S_data[:, j_idx] = (Qf_p - Qf_base) / delta_mvar

    return pd.DataFrame(S_data, index=BRANCH_NAMES, columns=eval_buses)

S_auto = compute_sensitivity(build_ppc(), EVALUATED_BUSES, delta_mvar=1.0)
print('dFq/dQ Sensitivity Matrix (auto):')
display(S_auto.round(6))


## 6 - Article Matrix (Table 2 - for comparison)

In [ ]:
S_article = pd.DataFrame([
    [-0.8008, -0.3857, -0.2384, -0.2514, -0.2485, -0.2552, -0.2555],
    [-0.1677, -0.5034, -0.3104, -0.3273, -0.3230, -0.3309, -0.3312],
    [-0.0372, -0.1116, -0.3464, -0.3652, -0.3600, -0.3681, -0.3685],
    [-0.0372, -0.1116, -0.3464, -0.3652, -0.3600, -0.3681, -0.3685],
    [ 0.2090, -0.3808, -0.2350, -0.2478, -0.2447, -0.2508, -0.2511],
    [ 0.0083,  0.0251, -0.1077, -0.1136, -0.1120, -0.1146, -0.1148],
    [ 0.0083,  0.0251, -0.1077, -0.1136, -0.1120, -0.1146, -0.1148],
    [ 0.0083,  0.0251, -0.1077, -0.1136, -0.1120, -0.1146, -0.1148],
    [ 0.0083,  0.0251, -0.1077, -0.1136, -0.1120, -0.1146, -0.1148],
    [ 0.0083,  0.0251, -0.1077, -0.1136, -0.1120, -0.1146, -0.1148],
    [-0.0054, -0.0162, -0.0502, -1.1074, -0.0522, -0.0534, -0.0534],
    [-0.0074, -0.0222, -0.0690, -0.0728, -1.1097, -1.1330, -1.1343],
    [-0.0023, -0.0068, -0.0212, -0.0224, -0.0368, -1.0578, -1.0590],
    [-0.0005, -0.0016, -0.0050, -0.0053, -0.0086, -0.0127, -1.0139],
], index=BRANCH_NAMES, columns=EVALUATED_BUSES)

# Select matrix source: 'auto' uses PYPOWER result; 'article' uses Table 2
MATRIX_SOURCE = 'auto'

if MATRIX_SOURCE == 'article':
    S = S_article.copy()
    print('Using ARTICLE matrix.')
else:
    S = S_auto.copy()
    print('Using AUTO (PYPOWER) matrix.')

print('Selected sensitivity matrix:')
display(S.round(6))


## 7 - Heatmaps

In [ ]:
plt.rcParams.update({'font.family': 'serif', 'font.size': 10})

# 7a: Sensitivity heatmap
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    S, ax=ax, cmap='turbo', annot=True, fmt='.4f',
    linewidths=0.4, square=True,
    cbar_kws={'label': 'dFq/dQ (MVAR/MVAR)'},
    xticklabels=[f'Bus {b}' for b in S.columns],
    yticklabels=S.index,
)
ax.set_title('Sensitivity Matrix dFq/dQ - BPA 10-Bus', fontsize=12)
ax.set_xlabel('Bus (Qd injection)', fontsize=10)
ax.set_ylabel('Branch', fontsize=10)
plt.tight_layout()
fig.savefig(OUT / 'sensitivity_heatmap.pdf', dpi=600, bbox_inches='tight')
fig.savefig(OUT / 'sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved sensitivity_heatmap.pdf / .png')
plt.close(fig)

# 7b: |S_auto - S_article| difference heatmap
diff = (S_auto - S_article).abs()
fig2, ax2 = plt.subplots(figsize=(9, 6))
sns.heatmap(
    diff, ax=ax2, cmap='turbo', annot=True, fmt='.4f',
    linewidths=0.4, square=True,
    cbar_kws={'label': '|AUTO - ARTICLE|'},
    xticklabels=[f'Bus {b}' for b in diff.columns],
    yticklabels=diff.index,
)
ax2.set_title('|S_auto - S_article| Difference Heatmap - BPA 10-Bus', fontsize=12)
ax2.set_xlabel('Bus', fontsize=10)
ax2.set_ylabel('Branch', fontsize=10)
plt.tight_layout()
fig2.savefig(OUT / 'sensitivity_diff_heatmap.pdf', dpi=600, bbox_inches='tight')
fig2.savefig(OUT / 'sensitivity_diff_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved sensitivity_diff_heatmap.pdf / .png')
plt.close(fig2)


## 8 - Clustering Functions

In [ ]:
def groups_for_branch(row, epsilon_percent, min_group_size=2):
    '''Compare sensitivities in one branch row; return list of bus-groups.
    Uses relative tolerance, but falls back to absolute tolerance when ref ~ 0.'''
    eps   = epsilon_percent / 100.0
    buses = list(row.index)
    vals  = row.values
    used  = [False] * len(buses)
    groups = []
    for i in range(len(buses)):
        if used[i]:
            continue
        ref = vals[i]
        tol = abs(ref) * eps if abs(ref) > 1e-9 else eps
        grp = [buses[i]]
        used[i] = True
        for j in range(i + 1, len(buses)):
            if used[j]:
                continue
            if abs(vals[j] - ref) <= tol:
                grp.append(buses[j])
                used[j] = True
        if len(grp) >= min_group_size:
            groups.append(tuple(sorted(grp)))
    return groups


def all_branch_groups(S_df, epsilon_percent):
    '''Return DataFrame: branch -> list of groups found at given tolerance.'''
    records = []
    for br, row in S_df.iterrows():
        grps = groups_for_branch(row, epsilon_percent)
        records.append({'Branch': br, 'Groups': grps})
    return pd.DataFrame(records).set_index('Branch')


def exact_support(S_df, epsilon_percent):
    '''For each unique group across all branches, compute exact support (%).'''
    _empty = pd.DataFrame(columns=['Exact_support_percent'])
    _empty.index.name = 'Group'
    all_grps = set()
    branch_grp_map = {}
    for br, row in S_df.iterrows():
        grps = groups_for_branch(row, epsilon_percent)
        branch_grp_map[br] = set(grps)
        all_grps.update(grps)
    if not all_grps:
        return _empty
    n_br = len(S_df)
    records = []
    for g in sorted(all_grps):
        count = sum(1 for br in S_df.index if g in branch_grp_map[br])
        records.append({'Group': str(g),
                        'Exact_support_percent': round(100.0 * count / n_br, 2)})
    df_out = pd.DataFrame(records).set_index('Group')
    return df_out


def inclusion_support(S_df, epsilon_percent, candidate_groups=None):
    '''For each candidate group, count branches where that group is a subset
    of any branch group (inclusion criterion). Return support (%).'''
    _empty = pd.DataFrame(columns=['Inclusion_support_percent'])
    _empty.index.name = 'Group'
    branch_grp_map = {}
    all_grps = set()
    for br, row in S_df.iterrows():
        grps = groups_for_branch(row, epsilon_percent)
        branch_grp_map[br] = grps
        all_grps.update(grps)
    if candidate_groups is None:
        candidate_groups = sorted(all_grps)
    if not candidate_groups:
        return _empty
    n_br = len(S_df)
    records = []
    for g in candidate_groups:
        g_set = set(g)
        count = sum(
            1 for br in S_df.index
            if any(g_set.issubset(set(bg)) for bg in branch_grp_map[br])
        )
        records.append({'Group': str(g),
                        'Inclusion_support_percent': round(100.0 * count / n_br, 2)})
    df_out = pd.DataFrame(records).set_index('Group')
    return df_out


def validate_range(S_df, start, stop, step, min_support_percent=50.0):
    '''Sweep epsilon from start to stop (inclusive); return summary DataFrame.'''
    tolerances = np.arange(start, stop + step, step)
    rows = []
    for eps in tolerances:
        es  = exact_support(S_df, eps)
        inc = inclusion_support(S_df, eps)
        col_es  = 'Exact_support_percent'
        col_inc = 'Inclusion_support_percent'
        n_exact = int((es[col_es]   >= min_support_percent).sum()) if col_es  in es.columns  else 0
        n_incl  = int((inc[col_inc] >= min_support_percent).sum()) if col_inc in inc.columns else 0
        rows.append({
            'Tolerance_%':         round(float(eps), 4),
            'N_groups_exact_sup':  n_exact,
            'N_groups_incl_sup':   n_incl,
        })
    return pd.DataFrame(rows).set_index('Tolerance_%')


print('Clustering functions defined.')


## 9 - Validation Sweep 0% to 10%

In [ ]:
MIN_SUPPORT_PERCENT = 50.0

sweep_summary = validate_range(S, 0, 10, 1, MIN_SUPPORT_PERCENT)
print('=== Sweep Summary (0% to 10%) ===')
display(sweep_summary)

# Store per-tolerance detail tables
detail_records = {}
for eps in range(0, 11):
    abg = all_branch_groups(S, eps)
    es  = exact_support(S, eps)
    inc = inclusion_support(S, eps)
    detail_records[eps] = {'groups': abg, 'exact': es, 'inclusion': inc}
    print(f'\n--- epsilon = {eps}% ---')
    print('Branch groups:')
    display(abg)
    print('Exact support:')
    display(es)
    print('Inclusion support:')
    display(inc)


## 10 - Support Curve Plot

In [ ]:
TRACKED_GROUPS = [
    (7, 8),
    (9, 10),
    (5, 9, 10),
    (8, 9, 10),
    (7, 8, 9, 10),
    (5, 8, 9, 10),
]

tols      = list(range(0, 11))
inc_curves = {g: [] for g in TRACKED_GROUPS}

for eps in tols:
    inc_df = inclusion_support(S, eps, candidate_groups=TRACKED_GROUPS)
    for g in TRACKED_GROUPS:
        key = str(g)
        val = inc_df.loc[key, 'Inclusion_support_percent'] if key in inc_df.index else 0.0
        inc_curves[g].append(float(val))

fig, ax = plt.subplots(figsize=(8, 5))
markers = ['o', 's', '^', 'D', 'v', 'P']
for g, mk in zip(TRACKED_GROUPS, markers):
    ax.plot(tols, inc_curves[g], marker=mk, label=str(g))

ax.axhline(MIN_SUPPORT_PERCENT, color='red', linestyle='--', linewidth=1.2,
           label=f'Min support = {MIN_SUPPORT_PERCENT:.0f}%')
ax.set_xlabel('Tolerance (%)', fontsize=11)
ax.set_ylabel('Inclusion Support (%)', fontsize=11)
ax.set_title('Group Inclusion Support vs. Tolerance - BPA 10-Bus', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
fig.savefig(OUT / 'support_curve.pdf', dpi=600, bbox_inches='tight')
fig.savefig(OUT / 'support_curve.png', dpi=150, bbox_inches='tight')
print('Saved support_curve.pdf / .png')
plt.close(fig)


## 11 - Minimum Observer Analysis

In [ ]:
def minimum_observer_analysis(S_df, eps, min_support_percent=50.0):
    '''Incrementally add branches as observers; track how clusters evolve.'''
    branches = list(S_df.index)
    results  = []
    for k in range(1, len(branches) + 1):
        sub = S_df.iloc[:k]
        inc = inclusion_support(sub, eps)
        passing = inc[inc['Inclusion_support_percent'] >= min_support_percent]
        results.append({
            'N_observers':     k,
            'Last_branch':     branches[k - 1],
            'N_groups_passing': len(passing),
            'Groups_passing':  ', '.join(passing.index.tolist()) if len(passing) else '--',
        })
    return pd.DataFrame(results).set_index('N_observers')


for eps_val in [5, 8]:
    print(f'\n=== Minimum Observer Analysis (epsilon = {eps_val}%) ===')
    obs_df = minimum_observer_analysis(S, eps_val, MIN_SUPPORT_PERCENT)
    display(obs_df)


## 12 - MATPOWER / VQ Analysis (optional - requires MATLAB + MATPOWER)

In [ ]:
import os, shutil

# Auto-detect MATLAB executable (check PATH first, then common install locations)
MATLAB_CANDIDATE_PATHS = [
    '/usr/local/MATLAB/R2024b/bin/matlab',
    '/usr/local/MATLAB/R2024a/bin/matlab',
    '/usr/local/MATLAB/R2023b/bin/matlab',
    '/Applications/MATLAB_R2024b.app/bin/matlab',
    '/Applications/MATLAB_R2024a.app/bin/matlab',
    r'C:\Program Files\MATLAB\R2024b\bin\matlab.exe',
    r'C:\Program Files\MATLAB\R2024a\bin\matlab.exe',
]

MATLAB_EXE = shutil.which('matlab')   # check system PATH first
if MATLAB_EXE is None:
    for _p in MATLAB_CANDIDATE_PATHS:
        if os.path.isfile(_p):
            MATLAB_EXE = _p
            break

MATLAB_AVAILABLE = MATLAB_EXE is not None
# Set MATPOWER_ROOT to the MATPOWER directory if it is NOT already on the MATLAB path.
# Leave as empty string if MATPOWER is already on the MATLAB path.
MATPOWER_ROOT = ''

print(f'MATLAB executable : {MATLAB_EXE}')
print(f'MATLAB available  : {MATLAB_AVAILABLE}')
print(f'MATPOWER_ROOT     : "{MATPOWER_ROOT}" (empty = already on MATLAB path)')


### 12a - Write caseBPA.m to disk

In [ ]:
# caseBPA.m content
CASE_BPA_CONTENT = (
    'function mpc = caseBPA\n'
    '%% BPA 10-bus system - auto-generated by Python notebook\n'
    "mpc.version = '2';\n"
    'mpc.baseMVA = 100;\n'
    '%% bus data\n'
    'mpc.bus = [\n'
    '  1  3      0      0  0     0  1  0.9800    0.0  500  1  1.1  0.9;\n'
    '  2  2      0      0  0     0  1  0.9646   -8.6  500  1  1.1  0.9;\n'
    '  3  2      0      0  0     0  1  0.9553  -26.8  500  1  1.1  0.9;\n'
    '  4  1      0      0  0     0  1  1.0874   -4.0  500  1  1.1  0.9;\n'
    '  5  1      0      0  0     0  1  1.0616  -12.4  500  1  1.1  0.9;\n'
    '  6  1      0      0  0   520  1  1.0196  -30.7  500  1  1.1  0.9;\n'
    '  7  1   2500    700  0   445  1  0.9362  -37.2  500  1  1.1  0.9;\n'
    '  8  1      0      0  0   445  1  0.9337  -36.8  500  1  1.1  0.9;\n'
    '  9  1      0      0  0     0  1  0.8850  -44.1  500  1  1.1  0.9;\n'
    ' 10  1   2500      0  0     0  1  0.9220  -46.4  500  1  1.1  0.9;\n'
    '];\n'
    '%% generator data\n'
    'mpc.gen = [\n'
    '  1  4219  1207  1600  -1000  0.9800  100  1  4737.1  10  0 0 0 0 0 0 0 0 0 0 0;\n'
    '  2  1736   725   725   -200  0.9646  100  1  2077.1  10  0 0 0 0 0 0 0 0 0 0 0;\n'
    '  3  1155   700   700   -100  0.9730  100  1  1438.7  10  0 0 0 0 0 0 0 0 0 0 0;\n'
    '];\n'
    '%% branch data\n'
    'mpc.branch = [\n'
    '  1  4  0.0000  0.0020  1.0000  5000  5000  5000  0.8857  0  1  -360  360;\n'
    '  2  5  0.0000  0.0045  1.0000  2200  2200  2200  0.8857  0  1  -360  360;\n'
    '  3  6  0.0000  0.0125  0.0000  1600  1600  1600  0.9024  0  1  -360  360;\n'
    '  3  6  0.0000  0.0125  0.0000  1600  1600  1600  0.9024  0  1  -360  360;\n'
    '  4  5  0.0000  0.0040  0.0000  5000  5000  5000  0.0000  0  1  -360  360;\n'
    '  5  6  0.0015  0.0288  1.1730  1100  1100  1100  0.0000  0  1  -360  360;\n'
    '  5  6  0.0015  0.0288  1.1730  1100  1100  1100  0.0000  0  1  -360  360;\n'
    '  5  6  0.0015  0.0288  1.1730  1100  1100  1100  0.0000  0  1  -360  360;\n'
    '  5  6  0.0015  0.0288  1.1730  1100  1100  1100  0.0000  0  1  -360  360;\n'
    '  5  6  0.0015  0.0288  1.1730  1100  1100  1100  0.0000  0  1  -360  360;\n'
    '  6  7  0.0000  0.0030  0.0000  3600  3600  3600  1.0664  0  1  -360  360;\n'
    '  6  8  0.0000  0.0026  0.0000  3500  3500  3500  1.0800  0  1  -360  360;\n'
    '  8  9  0.0010  0.0030  0.0000  3800  3800  3800  0.0000  0  1  -360  360;\n'
    '  9 10  0.0000  0.0010  0.0000  3700  3700  3700  0.9600  0  1  -360  360;\n'
    '];\n'
    '%% generator cost\n'
    'mpc.gencost = [\n'
    '  2  1500  0  3  0.1100  5.0  150;\n'
    '  2  2000  0  3  0.0850  1.2  600;\n'
    '  2  3000  0  3  0.1225  1.0  335;\n'
    '];\n'
    'end\n'
)

(OUT / 'caseBPA.m').write_text(CASE_BPA_CONTENT)
print('Written caseBPA.m')


### 12b - Write run_vq_matpower.m to disk

In [ ]:
VQ_SCRIPT_CONTENT = (
    'function run_vq_matpower()\n'
    '%% VQ curve analysis for BPA 10-bus - 4 scenarios\n'
    '%% This script is auto-generated by the Python notebook.\n'
    'if ~isempty(MATPOWER_PATH)\n'
    "    addpath(genpath(MATPOWER_PATH));\n"
    'end\n'
    'mpc0 = caseBPA;\n'
    "opt  = mpoption('verbose', 0, 'out.all', 0, 'pf.enforce_q_lims', 1);\n"
    'target_buses = [4, 5, 6, 7, 8, 9, 10];\n'
    'Q_sweep      = 0:1:30000;\n'
    'scenarios = struct();\n'
    "scenarios(1).name = 'S1_all_limited';\n"
    'scenarios(1).qmax = [1600; 725; 700];\n'
    'scenarios(1).qmin = [-1000; -200; -100];\n'
    "scenarios(2).name = 'S2_G1_8000';\n"
    'scenarios(2).qmax = [8000; 725; 700];\n'
    'scenarios(2).qmin = [-1000; -200; -100];\n'
    "scenarios(3).name = 'S3_RRB_unlimited';\n"
    'scenarios(3).qmax = [1600; 725; 1e9];\n'
    'scenarios(3).qmin = [-1000; -200; -1e9];\n'
    "scenarios(4).name = 'S4_all_unlimited';\n"
    'scenarios(4).qmax = [1e9; 1e9; 1e9];\n'
    'scenarios(4).qmin = [-1e9; -1e9; -1e9];\n'
    'summary_rows = {};\n'
    'trace_rows   = {};\n'
    'for s = 1:length(scenarios)\n'
    '    sc = scenarios(s);\n'
    '    for b = 1:length(target_buses)\n'
    '        tbus = target_buses(b);\n'
    '        bidx = find(mpc0.bus(:,1) == tbus);\n'
    '        Qd0  = mpc0.bus(bidx, 4);\n'
    '        V_critical = NaN;\n'
    '        Q_critical = NaN;\n'
    '        for qi = 1:length(Q_sweep)\n'
    '            mpc = mpc0;\n'
    '            mpc.gen(:,4) = sc.qmax;\n'
    '            mpc.gen(:,5) = sc.qmin;\n'
    '            mpc.bus(bidx, 4) = Qd0 + Q_sweep(qi);\n'
    '            r = runpf(mpc, opt);\n'
    '            if r.success\n'
    '                V_bus = r.bus(bidx, 8);\n'
    "                trace_rows{end+1} = {sc.name, tbus, Q_sweep(qi), V_bus};\n"
    '                V_critical = V_bus;\n'
    '                Q_critical = Q_sweep(qi);\n'
    '            else\n'
    '                break;\n'
    '            end\n'
    '        end\n'
    "        summary_rows{end+1} = {sc.name, tbus, Q_critical, V_critical};\n"
    '    end\n'
    'end\n'
    "fid = fopen('VQ_MATPOWER_summary.csv','w');\n"
    "fprintf(fid,'Scenario,Bus,Q_critical_MVAR,V_critical_pu\\n');\n"
    'for i = 1:length(summary_rows)\n'
    '    r = summary_rows{i};\n'
    "    fprintf(fid,'%s,%d,%.4f,%.6f\\n', r{1}, r{2}, r{3}, r{4});\n"
    'end\n'
    'fclose(fid);\n'
    "fid2 = fopen('VQ_MATPOWER_trace.csv','w');\n"
    "fprintf(fid2,'Scenario,Bus,Q_MVAR,V_pu\\n');\n"
    'for i = 1:length(trace_rows)\n'
    '    r = trace_rows{i};\n'
    "    fprintf(fid2,'%s,%d,%.4f,%.6f\\n', r{1}, r{2}, r{3}, r{4});\n"
    'end\n'
    'fclose(fid2);\n'
    "fprintf('VQ analysis complete. summary=%d rows, trace=%d rows\\n', ..."
    '        length(summary_rows), length(trace_rows));\n'
    'end\n'
)

# Replace the MATPOWER_PATH placeholder with the actual value at runtime
# The script uses the Python variable MATPOWER_ROOT passed as a literal string
VQ_SCRIPT_RUNTIME = VQ_SCRIPT_CONTENT.replace(
    'if ~isempty(MATPOWER_PATH)',
    f"MATPOWER_PATH = '{MATPOWER_ROOT}';\nif ~isempty(MATPOWER_PATH)"
)

(OUT / 'run_vq_matpower.m').write_text(VQ_SCRIPT_RUNTIME)
print('Written run_vq_matpower.m')


### 12c - Run MATLAB (subprocess)

In [ ]:
VQ_RUN_OK = False  # True only when MATLAB succeeds AND CSVs are non-empty

if not MATLAB_AVAILABLE:
    print('WARNING: MATLAB not found - skipping VQ MATPOWER execution.')
    print('         Install MATLAB + MATPOWER, set MATLAB_EXE and MATPOWER_ROOT, then re-run.')
else:
    try:
        out_abs = str(OUT.resolve())
        mp_path_cmd = ''
        if MATPOWER_ROOT:
            mp_path_cmd = f"addpath(genpath('{MATPOWER_ROOT}')); "
        matlab_cmd = (
            f"{mp_path_cmd}"
            f"cd('{out_abs}'); "
            'run_vq_matpower; exit;'
        )
        result = subprocess.run(
            [MATLAB_EXE, '-nodisplay', '-nosplash', '-nodesktop', '-r', matlab_cmd],
            capture_output=True, text=True, timeout=600,
        )
        log_out = OUT / 'matlab_stdout.log'
        log_err = OUT / 'matlab_stderr.log'
        log_out.write_text(result.stdout)
        log_err.write_text(result.stderr)
        print(f'Saved {log_out} and {log_err}')
        print('=== MATLAB stdout (last 3000 chars) ===')
        print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
        if result.stderr.strip():
            print('=== MATLAB stderr (last 2000 chars) ===')
            print(result.stderr[-2000:])
        csv_summary = OUT / 'VQ_MATPOWER_summary.csv'
        csv_trace   = OUT / 'VQ_MATPOWER_trace.csv'
        ok = (
            result.returncode == 0
            and csv_summary.exists() and csv_summary.stat().st_size > 10
            and csv_trace.exists()   and csv_trace.stat().st_size   > 10
        )
        VQ_RUN_OK = bool(ok)
        print(f'VQ_RUN_OK = {VQ_RUN_OK}')
    except Exception as _e:
        print(f'WARNING: MATLAB execution failed: {_e}')
        print('Continuing without VQ results.')

print(f'Final VQ_RUN_OK = {VQ_RUN_OK}')


### 12d - VQ Curve Plots (Python, from MATPOWER CSV results)

In [ ]:
if VQ_RUN_OK:
    df_summary = pd.read_csv(OUT / 'VQ_MATPOWER_summary.csv')
    df_trace   = pd.read_csv(OUT / 'VQ_MATPOWER_trace.csv')
    display(df_summary)

    scenarios_list = df_trace['Scenario'].unique()
    buses_list     = sorted(df_trace['Bus'].unique())
    n_sc   = len(scenarios_list)
    n_bus  = len(buses_list)

    fig, axes = plt.subplots(n_sc, n_bus, figsize=(3 * n_bus, 3 * n_sc),
                             sharex=False, sharey=False)
    if n_sc == 1:
        axes = axes[np.newaxis, :]
    if n_bus == 1:
        axes = axes[:, np.newaxis]

    for r_idx, sc in enumerate(scenarios_list):
        for c_idx, b in enumerate(buses_list):
            ax = axes[r_idx, c_idx]
            sub = df_trace[(df_trace['Scenario'] == sc) & (df_trace['Bus'] == b)]
            if not sub.empty:
                ax.plot(sub['Q_MVAR'], sub['V_pu'], linewidth=1.5)
            ax.set_title(f'{sc}\nBus {b}', fontsize=7)
            ax.set_xlabel('Q (MVAR)', fontsize=7)
            ax.set_ylabel('V (pu)', fontsize=7)
            ax.tick_params(labelsize=6)
            ax.grid(True, linestyle=':', alpha=0.5)

    plt.suptitle('VQ Curves - BPA 10-Bus System (MATPOWER)', fontsize=11, y=1.01)
    plt.tight_layout()
    fig.savefig(OUT / 'VQ_curves.pdf', dpi=600, bbox_inches='tight')
    fig.savefig(OUT / 'VQ_curves.png', dpi=150, bbox_inches='tight')
    print('Saved VQ_curves.pdf / .png')
    plt.close(fig)
else:
    print('VQ_RUN_OK is False - VQ plots skipped (MATLAB not available or failed).')


## 13 - Excel Export

In [ ]:
XLSX_PATH = OUT / 'BPA_hybrid_results.xlsx'

with pd.ExcelWriter(XLSX_PATH, engine='openpyxl') as xw:

    def _ws(df, name):
        df.to_excel(xw, sheet_name=name[:31])

    # Sensitivity matrices
    _ws(S_auto,    'S_auto_dFq_dQ')
    _ws(S_article, 'S_article_dFq_dQ')
    _ws(S,         'S_cluster_used')

    # Base power flow results
    _ws(df_bus,    'Base_Bus')
    _ws(df_gen,    'Base_Gen')
    _ws(df_branch, 'Base_Branch')

    # Sweep summary
    _ws(sweep_summary, 'Sweep_0_10pct')

    # Support curve data
    _support_records = []
    for _g in TRACKED_GROUPS:
        for _eps_v, _inc_v in zip(tols, inc_curves[_g]):
            _support_records.append({
                'Group':           str(_g),
                'Tolerance_%':     _eps_v,
                'Inclusion_sup_%': _inc_v,
            })
    _ws(pd.DataFrame(_support_records), 'Support_curve')

    # Observer analysis
    for _eps_val in [5, 8]:
        _obs_df = minimum_observer_analysis(S, _eps_val, MIN_SUPPORT_PERCENT)
        _ws(_obs_df, f'Observers_{_eps_val}pct')

    # Per-tolerance detail
    for _N in range(0, 11):
        _ws(detail_records[_N]['groups'],    f'Groups_{_N}pct')
        _ws(detail_records[_N]['exact'],     f'Exact_{_N}pct')
        _ws(detail_records[_N]['inclusion'], f'Inclusion_{_N}pct')

    # VQ results (only if MATLAB succeeded)
    if VQ_RUN_OK:
        _ws(df_summary, 'VQ_summary')
        _ws(df_trace,   'VQ_trace')

print(f'Excel workbook saved -> {XLSX_PATH}')
